#### 15 - Responsible AI Model Assessment

##### 1. Notebook Objective


This notebook performs a practical Responsible AI assessment of the
fine-tuned ResNet18 product-defect classifier.

The assessment focuses on evidence that can be measured using the current
dataset and model:

1. Reconstruct model predictions on the original test split.
2. Measure overall and per-class performance.
3. Quantify false negatives and false positives.
4. Analyze prediction confidence.
5. Identify high-confidence incorrect predictions.
6. Reassess evaluation integrity using exact duplicate detection.
7. Evaluate performance on the non-duplicate test subset.
8. Document what fairness and reliability questions cannot be answered
   with the available metadata.
9. Define model limitations and appropriate-use considerations.

The notebook does not attempt to produce a single Responsible AI score.
Instead, it reports multiple pieces of evidence that should be considered
together when assessing whether the system is appropriate for its intended
use.

##### 2. What are we assessing?

This notebook evaluates several Responsible AI dimensions:

| Dimension | Question |
|---|---|
| Performance | How accurately does the model classify the test images? |
| Error behavior | What kinds of mistakes does the model make? |
| Defect risk | How often are actual defects classified as OK? |
| Confidence | How confident is the model when correct and incorrect? |
| Evaluation integrity | Is the supplied test set independent of development data? |
| Explainability evidence | What did Grad-CAM reveal about selected failures? |
| Reliability | What reliability claims can the current dataset support? |
| Fairness / slices | Which subgroup analyses are possible with available metadata? |
| Transparency | What limitations must be communicated? |
| Human oversight | Where should human review remain part of the process? |

##### 3. Notebook Imports

In [0]:
import hashlib
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

from src.data_preparation import (
    create_image_dataloaders,
)
from src.responsible_ai import (
    collect_predictions,
    get_false_negative_indices,
    summarize_confidence,
)

##### 4. Configuration

In [0]:
RESNET18_RUN_ID = (
    "3f9b44ae55a449c5850f0b48c4ee4fa6"
)

RESNET18_MODEL_URI = (
    f"runs:/{RESNET18_RUN_ID}/model"
)

HIGH_CONFIDENCE_THRESHOLD = 0.90

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Model URI:", RESNET18_MODEL_URI)

##### 5. Load the persisted model

In [0]:
model = mlflow.pytorch.load_model(
    RESNET18_MODEL_URI
).to(device)

model.eval()

print(model.fc)

##### 6. Recreate the evaluation data

In [0]:
data = create_image_dataloaders(
    preprocessing_mode="pretrained"
)

print("Classes:", data.class_names)
print("Class mapping:", data.class_to_idx)

print(
    "Test images:",
    len(data.test_dataset),
)

In [0]:
defect_class_index = (
    data.class_to_idx["def_front"]
)

ok_class_index = (
    data.class_to_idx["ok_front"]
)

##### 7. Collect predictions

In [0]:
records = collect_predictions(
    model=model,
    data_loader=data.test_loader,
    device=device,
)

print(
    "Prediction records:",
    len(records),
)

``` text
One image
   ↓
ResNet18
   ↓
logits
   ↓
softmax
   ↓
probabilities
   ↓
predicted class + confidence

```

##### 8. Reconstruct overall performance

In [0]:
y_true = np.array(
    [
        record.actual_label
        for record in records
    ]
)

y_pred = np.array(
    [
        record.predicted_label
        for record in records
    ]
)

accuracy = np.mean(
    y_true == y_pred
)

print(
    f"Test Accuracy: {accuracy:.4f}"
)

In [0]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[
        defect_class_index,
        ok_class_index,
    ],
)

print(cm)

##### 9. Per-class metrics

In [0]:
report = classification_report(
    y_true,
    y_pred,
    target_names=data.class_names,
    digits=4,
)

print(report)

The most important defect metric is recall:

Defect recall ≈ 99.34%

Why?

Because:

Defect recall = correctly detected defects/all actual defects

In our case: 450/453 ≈ 99.34%

##### 11. ⚖️ RAI — quantify operationally important errors

In [0]:
missed_defects = cm[0, 1]
false_defect_alarms = cm[1, 0]

actual_defects = cm[0].sum()
actual_ok = cm[1].sum()

missed_defect_rate = (
    missed_defects
    / actual_defects
)

false_alarm_rate = (
    false_defect_alarms
    / actual_ok
)

print(
    "Missed defects:",
    missed_defects,
)

print(
    "Missed defect rate:",
    f"{missed_defect_rate:.2%}",
)

print(
    "False defect alarms:",
    false_defect_alarms,
)

print(
    "False alarm rate:",
    f"{false_alarm_rate:.2%}",
)

##### ⚖️ RAI Interpretation

- The ResNet18 classifier achieved strong overall test performance, but the
Responsible AI assessment should not stop at accuracy.

- Three actual defect images were classified as OK.

- In an automated quality-control workflow, these false negatives could be
more consequential than false defect alarms because a defective product
could potentially pass inspection.

- Whether a 0.66% observed missed-defect rate is acceptable cannot be decided
from the model metrics alone. The acceptable risk level must be defined
using manufacturing, quality, safety, and business requirements.

##### 11. Analyze model confidence

In [0]:
confidence_summary = summarize_confidence(
    records=records,
    high_confidence_threshold=(
        HIGH_CONFIDENCE_THRESHOLD
    ),
)

print(
    "Total predictions:",
    confidence_summary.total_predictions,
)

print(
    "Correct predictions:",
    confidence_summary.correct_predictions,
)

print(
    "Incorrect predictions:",
    confidence_summary.incorrect_predictions,
)

print(
    "Mean confidence:",
    confidence_summary.mean_confidence,
)

print(
    "Mean confidence when correct:",
    confidence_summary.mean_correct_confidence,
)

print(
    "Mean confidence when incorrect:",
    confidence_summary.mean_incorrect_confidence,
)

print(
    "High-confidence errors:",
    confidence_summary.high_confidence_errors,
)

##### 12. Visualize confidence: correct vs incorrect

In [0]:
correct_confidences = [
    record.confidence
    for record in records
    if (
        record.actual_label
        == record.predicted_label
    )
]

incorrect_confidences = [
    record.confidence
    for record in records
    if (
        record.actual_label
        != record.predicted_label
    )
]

plt.figure(
    figsize=(8, 5)
)

plt.boxplot(
    [
        correct_confidences,
        incorrect_confidences,
    ],
    tick_labels=[
        "Correct",
        "Incorrect",
    ],
)

plt.ylabel(
    "Maximum Softmax Probability"
)

plt.title(
    "Prediction Confidence: "
    "Correct vs Incorrect"
)

plt.show()

**⚖️ RAI interpretation**

The point isn't to prove: high confidence = correct

In fact, our project already demonstrated the opposite.

Instead we're asking:

How does confidence behave when the model succeeds versus when it fails?

##### 13. Inspect every false negative

In [0]:
false_negative_indices = (
    get_false_negative_indices(
        records=records,
        positive_class_index=(
            defect_class_index
        ),
    )
)

print(
    "False negatives:",
    len(false_negative_indices),
)

In [0]:
for index in false_negative_indices:

    record = records[index]

    image_path, _ = (
        data.test_dataset.samples[index]
    )

    print(
        "\nImage:",
        os.path.basename(image_path),
    )

    print(
        "Actual:",
        data.class_names[
            record.actual_label
        ],
    )

    print(
        "Predicted:",
        data.class_names[
            record.predicted_label
        ],
    )

    print(
        "Confidence:",
        f"{record.confidence:.2%}",
    )

    print(
        "Probabilities:",
        record.probabilities,
    )

##### 14. High-confidence error analysis

In [0]:
high_confidence_error_indices = [
    index
    for index, record
    in enumerate(records)
    if (
        record.actual_label
        != record.predicted_label
        and record.confidence
        >= HIGH_CONFIDENCE_THRESHOLD
    )
]

print(
    "High-confidence errors:",
    len(high_confidence_error_indices),
)

**⚖️ RAI Finding**

- At least one incorrect prediction was produced with very high model
confidence.

- This demonstrates that maximum softmax probability should not be interpreted
as a guarantee that the prediction is correct.

- Confidence may still be useful as one input to operational decision rules,
but any threshold-based human-review strategy should be validated against
real failure data rather than assuming that high-confidence predictions are
automatically safe.

##### 16. Confidence is not calibration


- The current analysis examines maximum softmax probabilities.

- This is not, by itself, a formal calibration assessment.

- A model is well calibrated when predictions assigned a particular
probability are correct at approximately the corresponding empirical rate.

- For example, among many predictions made with approximately 90% confidence,
a well-calibrated model would ideally be correct approximately 90% of the
time.

- Because the current test set contains only three ResNet18 errors, detailed
calibration conclusions would be unstable.

- Therefore, this notebook treats confidence as diagnostic information and
does not claim that the classifier is calibrated.

###### Excellent RAI principle: 

Not enough evidence → don't make the claim.

##### 17. Evaluation integrity — exact duplicate audit

In [0]:
def calculate_sha256(
    file_path: str,
) -> str:
    """
    Calculate the SHA-256 hash of a file.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        for chunk in iter(
            lambda: file.read(8192),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()

In [0]:
from src.project_config import RAW_DATA_DIR

development_dir = os.path.join(
    RAW_DATA_DIR,
    "train",
)

test_dir = os.path.join(
    RAW_DATA_DIR,
    "test",
)

##### 18. Build the development hash index

In [0]:
development_hashes = defaultdict(
    list
)

for root, _, files in os.walk(
    development_dir
):

    for filename in files:

        file_path = os.path.join(
            root,
            filename,
        )

        try:
            file_hash = calculate_sha256(
                file_path
            )
        except OSError:
            continue

        development_hashes[
            file_hash
        ].append(file_path)

In [0]:
duplicate_test_paths = set()

for root, _, files in os.walk(
    test_dir
):

    for filename in files:

        file_path = os.path.join(
            root,
            filename,
        )

        try:
            file_hash = calculate_sha256(
                file_path
            )
        except OSError:
            continue

        if file_hash in development_hashes:
            duplicate_test_paths.add(
                file_path
            )

In [0]:
print(
    "Exact development/test duplicates:",
    len(duplicate_test_paths),
)

##### 19. Why SHA-256 matters

SHA-256 compares the exact file bytes.

If two images have the same SHA-256 hash in this audit, they are treated as
exact byte-identical duplicates.

This test does not detect:

- near duplicates;
- differently compressed copies;
- resized copies;
- slightly altered copies;
- multiple images of the same physical casting;
- visually similar sequential captures.

Therefore:     No exact hash match
does not mean:     guaranteed independent physical sample.

This is why we'll call our later subset:

non-duplicate test subset

not:

clean test set.

##### 20. Map duplicate status to the test predictions

In [0]:
duplicate_test_paths_normalized = {
    os.path.normpath(path)
    for path in duplicate_test_paths
}

In [0]:
non_duplicate_mask = np.array(
    [
        os.path.normpath(path)
        not in duplicate_test_paths_normalized
        for path, _
        in data.test_dataset.samples
    ]
)

print(
    "Original test images:",
    len(non_duplicate_mask),
)

print(
    "Exact duplicates:",
    np.sum(
        ~non_duplicate_mask
    ),
)

print(
    "Non-duplicate images:",
    np.sum(
        non_duplicate_mask
    ),
)

##### 21. Recalculate performance on the non-duplicate subset

In [0]:
y_true_non_duplicate = (
    y_true[non_duplicate_mask]
)

y_pred_non_duplicate = (
    y_pred[non_duplicate_mask]
)

non_duplicate_accuracy = np.mean(
    y_true_non_duplicate
    == y_pred_non_duplicate
)

non_duplicate_cm = confusion_matrix(
    y_true_non_duplicate,
    y_pred_non_duplicate,
    labels=[
        defect_class_index,
        ok_class_index,
    ],
)

print(
    "Non-duplicate accuracy:",
    f"{non_duplicate_accuracy:.4f}",
)

print(
    "\nNon-duplicate confusion matrix:"
)

print(
    non_duplicate_cm
)

##### 22. Compare original and non-duplicate evaluation

In [0]:
print(
    "Original test accuracy:",
    f"{accuracy:.2%}",
)

print(
    "Non-duplicate subset accuracy:",
    f"{non_duplicate_accuracy:.2%}",
)

##### ⚖️ RAI Finding — Evaluation Integrity

The supplied 715-image test folder is not completely independent of the
development data.

The exact SHA-256 audit identified 64 byte-identical images shared between
the original development pool and test folder.

After excluding those exact duplicates, 651 test images remained.

ResNet18 performance changed only slightly:

- Original test accuracy: approximately 99.58%
- Non-duplicate subset accuracy: approximately 99.54%

All three ResNet18 false negatives remained in the non-duplicate subset.

Therefore, exact development/test duplication is an important evaluation
integrity limitation that must be documented, but it does not appear to
explain the model's high measured classification performance.

The hash audit does not rule out near duplicates, repeated views of the same
physical product, or other forms of sample dependence.

##### 23. 🔍 XAI evidence from Notebook 13


Notebook 13 analyzed the three non-duplicate false-negative defect images
using Grad-CAM.

Separate explanations were generated for:

- the predicted `ok_front` class;
- the `def_front` class.

The explanations showed broad spatial regions influencing each class score.

Some repeated spatial tendencies were observed across the failure cases,
including influence in upper casting regions for the predicted OK class and
right-side/background influence in some defect-class explanations.

However:

- only three ResNet18 failure cases were available;
- Layer4 Grad-CAM has coarse 7 × 7 spatial resolution;
- the dataset does not contain ground-truth defect localization labels;
- Grad-CAM does not establish causal model reliance.

These findings therefore generate reliability hypotheses rather than proving
that the classifier relies on a specific shortcut.

##### 24. What fairness analysis can we perform?

Responsible AI assessments often compare model performance across relevant
groups or operational conditions.

For this industrial vision system, useful slices could include:

- manufacturing machine;
- production line;
- casting batch;
- camera;
- lighting condition;
- product variant;
- defect subtype;
- operator or inspection station;
- acquisition date.

The current dataset does not provide sufficient metadata for these
operational slices.

Therefore, this project cannot responsibly claim that the model performs
equally across different machines, cameras, lighting conditions, production
batches, or defect subtypes.

The absence of subgroup metadata is itself a limitation of the current
Responsible AI assessment.

This is a very important professional habit:

``` text

Missing data
    ↓
cannot evaluate question
    ↓
document limitation

NOT

Missing data
    ↓
assume everything is fine

```

##### 25. Reliability assessment


##### Evidence available

The current evaluation provides evidence that:

- ResNet18 performs strongly on the supplied casting-image dataset.
- Performance remains similar after exact development/test duplicates are
  removed.
- Three non-duplicate defect images were incorrectly classified as OK.
- At least one incorrect prediction had very high model confidence.
- Grad-CAM identified broad regions influencing those predictions.

##### Evidence not available

The current dataset does not establish reliability under:

- different cameras;
- different lighting conditions;
- different image resolutions;
- new manufacturing machines;
- new production lines;
- new casting variants;
- unseen defect types;
- long-term production drift.

Therefore, strong test performance should not be interpreted as proof of
reliability under all production conditions.

##### 26. Appropriate use versus unsupported use

##### Appropriate Use

Based on the current project evidence, the model can be treated as a
prototype for classifying casting-front images into:

- `def_front`
- `ok_front`

under conditions reasonably similar to the available dataset.

##### Unsupported Claims

The current evidence does not establish that the model can:

- locate the exact physical defect;
- identify specific defect subtypes;
- generalize to arbitrary casting products;
- remain reliable under different cameras or lighting conditions;
- replace all human quality-control decisions;
- guarantee that a high-confidence OK prediction is truly defect-free.

##### 27. Human oversight


A simple confidence-only escalation policy would be insufficient for this
model.

One known false-negative defect prediction received approximately 98.46%
confidence for the `ok_front` class.

Therefore, a rule such as:
``` text

    low confidence
        ↓
    human inspection

    high confidence
        ↓
    automatically accept

```

could allow some high-confidence model errors to bypass human review.

Any production review policy should therefore be designed using domain risk,
failure analysis, process controls, and additional validation rather than
softmax confidence alone.

For high-consequence quality decisions, human inspection or another
independent control may remain appropriate until production reliability has
been demonstrated.

##### 28. Responsible AI assessment summary

## Responsible AI Assessment Summary

| Area | Evidence | Current Assessment |
|---|---|---|
| Overall performance | ~99.58% original test accuracy | Strong dataset-level performance |
| Defect recall | ~99.34% | 3 of 453 defects missed |
| False positives | 0 on original test | No observed OK→defect errors |
| High-confidence errors | Present | Confidence alone is insufficient for trust |
| Exact duplicate integrity | 64 / 715 test images overlap development pool | Evaluation limitation |
| Non-duplicate performance | ~99.54% on 651 images | Exact duplicates do not appear to explain high performance |
| XAI | Grad-CAM analyzed failures | Diagnostic evidence, not localization or causal proof |
| Calibration | Not established | More evidence required |
| Operational slice fairness | Metadata unavailable | Cannot assess |
| Camera/lighting robustness | Not tested | Cannot claim |
| Production drift | Not tested | Requires monitoring after deployment |
| Human oversight | Still relevant | Confidence-only automation is insufficient |

##### 29. Key Learnings


1. Responsible AI assessment requires multiple forms of evidence rather than
   a single score.

2. Overall accuracy should be supplemented with per-class metrics and
   operationally meaningful error analysis.

3. The ResNet18 classifier missed three of 453 actual defect images in the
   original test split.

4. False negatives are especially important in this use case because they
   represent defects classified as OK.

5. Prediction confidence does not guarantee correctness.

6. At least one false-negative defect received very high `ok_front`
   confidence, demonstrating that confidence-only human-review policies can
   miss important failures.

7. Maximum softmax probability is not equivalent to calibrated probability.

8. Exact duplicate auditing identified 64 test images shared with the
   original development pool.

9. Removing exact development/test duplicates changed ResNet18 accuracy only
   slightly, from approximately 99.58% to 99.54%.

10. All three ResNet18 false negatives remained in the non-duplicate test
    subset.

11. Exact duplicate detection does not rule out near duplicates, repeated
    physical products, or other forms of sample dependence.

12. Grad-CAM provides useful diagnostic evidence but does not prove causal
    reliance or identify verified defect locations.

13. Responsible AI includes recognizing what cannot be evaluated with the
    available data.

14. The current dataset does not provide sufficient operational metadata to
    evaluate performance across machines, cameras, lighting conditions,
    batches, or defect subtypes.

15. Strong dataset-level test performance does not establish production
    reliability under distribution shift.

16. Human oversight and independent quality controls may remain necessary
    when model errors have meaningful consequences.

17. Responsible AI requires documenting intended use, unsupported claims,
    known failure modes, data limitations, and monitoring requirements.

##### 30. Notebook Conclusion


- This notebook performed a practical Responsible AI assessment of the
fine-tuned ResNet18 product-defect classifier.

- The model demonstrated strong classification performance on the supplied
test data, with approximately 99.58% accuracy and 99.34% recall for the
defect class. However, three actual defect images were classified as OK,
demonstrating that strong aggregate performance does not eliminate
operationally important failures.

- Confidence analysis further showed that an incorrect prediction can occur
with very high model confidence. Therefore, softmax confidence should not
be interpreted as a guarantee of correctness or used as the sole basis for
automated acceptance or human-review decisions.

- The evaluation-integrity audit identified 64 exact byte-identical images
shared between the original development pool and supplied test folder.
After removing these exact duplicates, ResNet18 accuracy remained
approximately 99.54%, and all three false-negative defects remained.
Therefore, exact duplication is an important limitation that must be
documented, but it does not appear to explain the model's strong measured
performance.

- Explainability evidence from Grad-CAM provided additional information about
broad spatial regions influencing selected class scores. These explanations
are useful for generating hypotheses about model behavior but do not provide
ground-truth defect localization or prove causal reliance on particular
image regions.

- Several important Responsible AI questions cannot be answered using the
current dataset. In particular, the available data does not provide
sufficient metadata to assess reliability across cameras, lighting
conditions, manufacturing machines, production batches, product variants,
or defect subtypes.

- The model should therefore be treated as a strong experimental classifier
for images similar to the current dataset rather than as evidence of
universal production reliability.

- Responsible deployment would require additional production-representative
validation, appropriate human oversight and quality controls, monitoring for
distribution shift and performance degradation, model versioning,
auditability, and clearly documented intended-use limitations.


``` text

13 — EXPLAINABLE AI
────────────────────────
Why did the model produce
this particular prediction?

          ↓

Grad-CAM
activations
gradients
class-specific influence
failure explanations


14–15 — RESPONSIBLE AI
────────────────────────
Should this system be trusted
and used for this purpose?

          ↓

data integrity
evaluation integrity
error consequences
confidence behavior
reliability
fairness/slices
limitations
human oversight
monitoring
accountability

```


``` text

99.58% accuracy
       ↓
looks excellent

BUT

3 missed defects
1 highly confident error
64 exact development/test duplicates
unknown camera robustness
unknown lighting robustness
no operational subgroup metadata

       ↓

Responsible AI asks us
to consider ALL of this evidence.

```